# 🔧 Feature Engineering — NTB Predictive Model
**Input:**   
**Output:**   
Builds all lag, rolling, momentum, demand, term-structure, and calendar features.

In [ ]:
import sys, pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
import warnings; warnings.filterwarnings("ignore")
sys.path.insert(0, "..")
from src.data_loader import load_clean
from src.features import engineer, get_feature_cols

plt.rcParams.update({"figure.dpi":120,"axes.spines.top":False,"axes.spines.right":False,"axes.grid":True,"grid.alpha":0.3})
print("Imports OK ✓")

## 1. Load Clean Data

In [ ]:
df = load_clean("../data/raw/Primary_Market_in_Excel.xlsx")
print(f"Shape: {df.shape}")
print(f"Date range: {df[chr(39)]auctionDate[chr(39)].min().date()} → {df[chr(39)]auctionDate[chr(39)].max().date()}")
df[["auctionDate","tenor_days","yield_pct","subscription_ratio"]].head()

## 2. Engineer Features for All Tenors

In [ ]:
feats = {}
for tenor in [91, 182, 364]:
    f = engineer(df, tenor)
    feats[tenor] = f
    cols = get_feature_cols(f)
    print(f"{tenor}-Day → {f.shape[0]} samples | {len(cols)} features")

feat364 = feats[364]
fcols   = get_feature_cols(feat364)
feat364.to_csv("../data/processed/tbill_features.csv", index=False)
print("
Saved → data/processed/tbill_features.csv")

## 3. Feature Distributions

In [ ]:
lag_cols = [c for c in fcols if "lag" in c]
fig, axes = plt.subplots(2, 3, figsize=(14,7))
for ax, col in zip(axes.flatten(), lag_cols[:6]):
    ax.hist(feat364[col].dropna(), bins=25, color="#1f77b4", edgecolor="white", alpha=0.8)
    ax.set_title(col, fontsize=10)
    ax.set_xlabel("Yield (%)")
plt.suptitle("Lag Feature Distributions — 364-Day NTB", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../data/processed/fig_feat_distributions.png", dpi=150)
plt.show()

## 4. Feature–Target Correlations

In [ ]:
corr = feat364[fcols + ["target"]].corr()["target"].drop("target").sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 9))
colors  = ["#2ca02c" if v > 0 else "#d62728" for v in corr.values]
ax.barh(corr.index, corr.values, color=colors, alpha=0.8, edgecolor="none")
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Feature Correlation with Target (Next Yield)
364-Day NTB",
             fontsize=12, fontweight="bold")
ax.set_xlabel("Pearson Correlation")
plt.tight_layout()
plt.savefig("../data/processed/fig_feat_target_corr.png", dpi=150)
plt.show()

print("Top positive predictors:")
print(corr.tail(6).round(4).to_string())
print("
Top negative predictors:")
print(corr.head(6).round(4).to_string())

## 5. Rolling Mean vs Actual — Signal Quality Check

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(feat364["auctionDate"], feat364["yield_pct"],  label="Actual Yield",      alpha=0.6, linewidth=1)
ax.plot(feat364["auctionDate"], feat364["roll_mean_3"], label="3-Auction MA",       linewidth=1.3, linestyle="--")
ax.plot(feat364["auctionDate"], feat364["roll_mean_6"], label="6-Auction MA",       linewidth=1.3, linestyle=":")
ax.plot(feat364["auctionDate"], feat364["target"],      label="Target (next yield)",linewidth=1.0, alpha=0.5, color="red")
ax.set_title("Actual vs Rolling Means vs Target — 364-Day NTB", fontweight="bold")
ax.set_ylabel("Yield (%)")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig("../data/processed/fig_signal_quality.png", dpi=150)
plt.show()

## 6. Subscription Ratio as a Predictor

In [ ]:
valid = feat364[["sub_ratio_lag1","target","auctionDate"]].dropna()
valid = valid[valid["sub_ratio_lag1"] < 30]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].scatter(valid["sub_ratio_lag1"], valid["target"], alpha=0.3, s=15, color="#ff7f0e")
z = np.polyfit(valid["sub_ratio_lag1"], valid["target"], 1)
xp = np.linspace(valid["sub_ratio_lag1"].min(), valid["sub_ratio_lag1"].max(), 100)
axes[0].plot(xp, np.polyval(z, xp), color="black", linewidth=1.5, label=f"Trend")
axes[0].set_xlabel("Subscription Ratio (lagged 1 auction)")
axes[0].set_ylabel("Next Yield (%)")
axes[0].set_title("Demand Pressure vs Next Yield")
axes[0].legend()

axes[1].plot(valid["auctionDate"], valid["sub_ratio_lag1"], color="#ff7f0e", linewidth=1)
axes[1].set_title("Subscription Ratio Over Time")
axes[1].set_ylabel("Ratio (x)")
axes[1].set_ylim(0, 20)

plt.suptitle("Subscription Ratio Analysis — 364-Day NTB", fontweight="bold")
plt.tight_layout()
plt.savefig("../data/processed/fig_sub_ratio.png", dpi=150)
plt.show()

## 7. Final Feature Summary

In [ ]:
print("="*55)
print("FEATURE ENGINEERING SUMMARY")
print("="*55)
print(f"
364-Day NTB dataset: {feat364.shape[0]} samples, {len(fcols)} features")
print(f"
Feature groups:")
groups = {
    "Lag features"      : [c for c in fcols if "lag" in c and "sub" not in c and "allot" not in c],
    "Rolling stats"     : [c for c in fcols if "roll" in c or "mean" in c or "std" in c],
    "Momentum"          : [c for c in fcols if "moment" in c or "roc" in c],
    "Demand (auction)"  : [c for c in fcols if "sub" in c or "allot" in c],
    "Term structure"    : [c for c in fcols if "spread" in c or "yield_9" in c or "yield_1" in c or "yield_3" in c],
    "Calendar"          : [c for c in fcols if c in ["month","quarter","year"]],
}
for grp_name, grp_cols in groups.items():
    print(f"  {grp_name:<22}: {grp_cols}")

top5 = feat364[fcols + ["target"]].corr()["target"].drop("target").abs().nlargest(5)
print(f"
Top 5 predictors by |correlation|:")
print(top5.round(4).to_string())